# FD004 Pipeline Code

## Preprocessing

In [ ]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 30
RUL_CAP = 125


def load_fd004() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Load raw train, test, and final RUL files.
    cols = ["unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3"] + [f"sensor_{i}" for i in range(1, 22)]
    train = pd.read_csv(RAW_DIR / "train_FD004.txt", sep=r"\s+", header=None, names=cols)
    test = pd.read_csv(RAW_DIR / "test_FD004.txt", sep=r"\s+", header=None, names=cols)
    rul = pd.read_csv(RAW_DIR / "RUL_FD004.txt", sep=r"\s+", header=None, names=["final_rul"])
    return train, test, rul


def add_labels_and_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str]]:
    # Drop mostly flat/noisy channels used in this project setup.
    drop_cols = [
        "op_setting_3",
        "sensor_1", "sensor_5", "sensor_6", "sensor_10", "sensor_16", "sensor_18", "sensor_19",
    ]
    train_df = train_df.drop(columns=drop_cols).copy()
    test_df = test_df.drop(columns=drop_cols).copy()

    sensor_cols = [c for c in train_df.columns if c.startswith("sensor_")]

    # Normalize cycle index per engine so all engines are on a similar scale.
    train_df["cycle_norm"] = train_df["cycle"] / train_df.groupby("unit_id")["cycle"].transform("max")
    test_df["cycle_norm"] = test_df["cycle"] / test_df.groupby("unit_id")["cycle"].transform("max")

    # Build capped RUL target for training.
    max_cycle = train_df.groupby("unit_id")["cycle"].max().rename("max_cycle")
    train_df = train_df.join(max_cycle, on="unit_id")
    train_df["RUL_raw"] = train_df["max_cycle"] - train_df["cycle"]
    train_df["RUL"] = train_df["RUL_raw"].clip(upper=RUL_CAP)
    train_df = train_df.drop(columns=["max_cycle"])

    feature_cols = sensor_cols + ["cycle_norm"]
    return train_df, test_df, sensor_cols, feature_cols


def scale_features(train_df: pd.DataFrame, test_df: pd.DataFrame, sensor_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Fit scaler on train sensors, reuse on test sensors.
    scaler = MinMaxScaler()
    train_df[sensor_cols] = scaler.fit_transform(train_df[sensor_cols].astype(float))
    test_df[sensor_cols] = scaler.transform(test_df[sensor_cols].astype(float))

    with open(PROCESSED_DIR / "feature_scaler.pkl", "wb") as fp:
        pickle.dump(scaler, fp)

    return train_df, test_df


def build_train_sequences(df: pd.DataFrame, feature_cols: list[str], seq_len: int):
    # Build sliding windows so each sample is one sequence.
    X, y, unit_ids = [], [], []
    for unit_id in df["unit_id"].unique():
        engine = df[df["unit_id"] == unit_id].sort_values("cycle")
        features = engine[feature_cols].to_numpy(dtype=np.float32)
        rul = engine["RUL"].to_numpy(dtype=np.float32)

        for i in range(len(features) - seq_len + 1):
            X.append(features[i:i + seq_len])
            y.append(rul[i + seq_len - 1])
            unit_ids.append(unit_id)

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32), np.asarray(unit_ids, dtype=np.int32)


def build_test_last_sequences(df: pd.DataFrame, rul_table: pd.DataFrame, feature_cols: list[str], seq_len: int):
    # For test set, keep only the last sequence of each engine.
    X_test, y_test, ids = [], [], []
    final_rul_map = {i + 1: float(v) for i, v in enumerate(rul_table["final_rul"].tolist())}

    for unit_id in df["unit_id"].unique():
        engine = df[df["unit_id"] == unit_id].sort_values("cycle")
        features = engine[feature_cols].to_numpy(dtype=np.float32)

        # Pad short histories up to sequence length.
        if len(features) < seq_len:
            pad = np.repeat(features[[0]], seq_len - len(features), axis=0)
            features = np.vstack([pad, features])

        X_test.append(features[-seq_len:])
        y_test.append(min(final_rul_map[unit_id], RUL_CAP))
        ids.append(unit_id)

    return np.asarray(X_test, dtype=np.float32), np.asarray(y_test, dtype=np.float32), np.asarray(ids, dtype=np.int32)


def sequence_balance(X: np.ndarray, y: np.ndarray, unit_ids: np.ndarray):
    # Downsample capped-RUL samples to reduce class imbalance.
    high_mask = y == RUL_CAP
    high_idx = np.where(high_mask)[0]
    low_idx = np.where(~high_mask)[0]

    rng = np.random.default_rng(42)
    keep_high_n = int(len(high_idx) * 0.3)
    keep_high_idx = rng.choice(high_idx, size=keep_high_n, replace=False) if keep_high_n > 0 else high_idx

    keep_idx = np.concatenate([low_idx, keep_high_idx])
    rng.shuffle(keep_idx)
    return X[keep_idx], y[keep_idx], unit_ids[keep_idx], len(high_idx), int(np.sum(y[keep_idx] == RUL_CAP))


def build_test_trajectories(test_df: pd.DataFrame, rul_df: pd.DataFrame, feature_cols: list[str]):
    # Build full per-cycle trajectories for curve plotting and inference demos.
    final_rul_map = {i + 1: float(v) for i, v in enumerate(rul_df["final_rul"].tolist())}
    trajectories = {}

    for unit_id in test_df["unit_id"].unique():
        engine = test_df[test_df["unit_id"] == unit_id].sort_values("cycle").copy()
        max_obs = int(engine["cycle"].max())
        engine["actual_rul"] = ((max_obs - engine["cycle"]) + final_rul_map[unit_id]).clip(upper=RUL_CAP)

        features = engine[feature_cols].to_numpy(dtype=np.float32)
        actual = engine["actual_rul"].to_numpy(dtype=np.float32)
        cycles = engine["cycle"].to_numpy(dtype=np.int32)

        seqs, actuals, cycle_list = [], [], []
        for i in range(len(engine)):
            window = features[max(0, i - SEQ_LEN + 1): i + 1]
            if len(window) < SEQ_LEN:
                pad = np.repeat(window[[0]], SEQ_LEN - len(window), axis=0)
                window = np.vstack([pad, window])
            seqs.append(window.astype(np.float32))
            actuals.append(float(actual[i]))
            cycle_list.append(int(cycles[i]))

        trajectories[int(unit_id)] = {
            "cycles": cycle_list,
            "actual_rul": actuals,
            "sequences": np.asarray(seqs, dtype=np.float32),
            "final_rul": float(min(final_rul_map[unit_id], RUL_CAP)),
        }

    return trajectories


def main() -> None:
    train_df, test_df, rul_df = load_fd004()
    train_df, test_df, sensor_cols, feature_cols = add_labels_and_features(train_df, test_df)
    train_df, test_df = scale_features(train_df, test_df, sensor_cols)

    X_train, y_train, train_unit_ids = build_train_sequences(train_df, feature_cols, SEQ_LEN)
    X_train, y_train, train_unit_ids, high_before, high_after = sequence_balance(X_train, y_train, train_unit_ids)

    X_test, y_test, test_unit_ids = build_test_last_sequences(test_df, rul_df, feature_cols, SEQ_LEN)

    # Save all processed arrays and metadata for later stages.
    np.save(PROCESSED_DIR / "X_train_sequences.npy", X_train)
    np.save(PROCESSED_DIR / "y_train_sequences.npy", y_train)
    np.save(PROCESSED_DIR / "train_sequence_unit_ids.npy", train_unit_ids)
    np.save(PROCESSED_DIR / "X_test_last.npy", X_test)
    np.save(PROCESSED_DIR / "y_test_last.npy", y_test)
    np.save(PROCESSED_DIR / "test_unit_ids.npy", test_unit_ids)

    with open(PROCESSED_DIR / "feature_columns.json", "w", encoding="utf-8") as fp:
        json.dump(feature_cols, fp, indent=2)

    with open(PROCESSED_DIR / "dataset_config.json", "w", encoding="utf-8") as fp:
        json.dump({"dataset": "FD004", "seq_len": SEQ_LEN, "rul_cap": RUL_CAP}, fp, indent=2)

    trajectories = build_test_trajectories(test_df, rul_df, feature_cols)
    with open(PROCESSED_DIR / "test_engine_trajectories.pkl", "wb") as fp:
        pickle.dump(trajectories, fp)

    sample_count = min(100, len(X_test))
    with open(PROCESSED_DIR / "test_sequences_sample.json", "w", encoding="utf-8") as fp:
        json.dump({"samples": X_test[:sample_count].tolist()}, fp)

    print("FD004 preprocessing complete")
    print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
    print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")
    print(f"RUL=125 sequences before/after balancing: {high_before}/{high_after}")
    print(f"Feature count: {len(feature_cols)}")


if __name__ == "__main__":
    main()

## Model

In [ ]:
import torch
import torch.nn as nn


class LSTMRULModel(nn.Module):
    # Basic LSTM regressor to predict a single RUL value.
    def __init__(self, input_size: int, hidden_size: int = 128, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Use final timestep representation for regression.
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.head(self.dropout(last)).squeeze(-1)


def mc_dropout_predict(model: nn.Module, x: torch.Tensor, n_passes: int = 50):
    # Keep dropout active during inference for uncertainty estimation.
    model.train()
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            preds.append(model(x).cpu())

    stacked = torch.stack(preds, dim=0)
    return stacked.mean(dim=0), stacked.std(dim=0)

## Training

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path(__file__).resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from model.lstm_model import LSTMRULModel


PROCESSED_DIR = Path("data/processed")
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cpu")
RUL_CAP = 125.0
EPOCHS = 60
PATIENCE = 10


def weighted_mse(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
    # Higher weight for low-RUL points.
    weights = 1.0 / (y_true + 0.1)
    return torch.mean(weights * (y_pred - y_true) ** 2)


def main() -> None:
    torch.manual_seed(42)
    np.random.seed(42)

    # Load prepared sequence dataset.
    X_all = np.load(PROCESSED_DIR / "X_train_sequences.npy")
    y_all = np.load(PROCESSED_DIR / "y_train_sequences.npy")
    seq_unit_ids = np.load(PROCESSED_DIR / "train_sequence_unit_ids.npy")

    input_size = X_all.shape[-1]

    # Split by engine ids so same engine does not leak across splits.
    unique_units = np.unique(seq_unit_ids)
    train_units, val_units = train_test_split(unique_units, test_size=0.2, random_state=42)

    train_mask = np.isin(seq_unit_ids, train_units)
    val_mask = np.isin(seq_unit_ids, val_units)

    X_train, y_train = X_all[train_mask], y_all[train_mask]
    X_val, y_val = X_all[val_mask], y_all[val_mask]

    print(f"Train split: {X_train.shape} | {y_train.shape}")
    print(f"Val split  : {X_val.shape} | {y_val.shape}")

    y_train_norm = y_train / RUL_CAP
    y_val_norm = y_val / RUL_CAP

    # Quick sanity run: overfit tiny subset to verify pipeline is trainable.
    tiny_n = min(50, len(X_train))
    X_small = torch.tensor(X_train[:tiny_n], dtype=torch.float32, device=DEVICE)
    y_small = torch.tensor(y_train_norm[:tiny_n], dtype=torch.float32, device=DEVICE)

    sanity_model = LSTMRULModel(input_size=input_size, hidden_size=128, num_layers=2, dropout=0.0).to(DEVICE)
    sanity_opt = torch.optim.Adam(sanity_model.parameters(), lr=0.005)
    sanity_criterion = nn.MSELoss()

    for epoch in range(1, 301):
        sanity_model.train()
        sanity_opt.zero_grad()
        pred = sanity_model(X_small)
        loss = sanity_criterion(pred, y_small)
        loss.backward()
        sanity_opt.step()

        if epoch in (1, 75, 150, 225, 300):
            print(f"Sanity epoch {epoch:03d} | MSE: {loss.item():.6f}")

    with torch.no_grad():
        sanity_preds = sanity_model(X_small).cpu().numpy() * RUL_CAP

    print(f"Sanity final MSE: {loss.item():.6f}")
    print(f"Sanity prediction STD: {np.std(sanity_preds):.4f}")

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_norm, dtype=torch.float32)),
        batch_size=128,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val_norm, dtype=torch.float32)),
        batch_size=128,
        shuffle=False,
    )

    model = LSTMRULModel(input_size=input_size, hidden_size=128, num_layers=2, dropout=0.2).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_rmse_hist, val_rmse_hist = [], []
    best_val_rmse = float("inf")
    no_improve = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_preds_epoch = []
        train_true_epoch = []

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            preds = model(xb)
            loss = weighted_mse(preds, yb)
            loss.backward()
            optimizer.step()

            train_preds_epoch.append(preds.detach().cpu().numpy() * RUL_CAP)
            train_true_epoch.append(yb.detach().cpu().numpy() * RUL_CAP)

        train_preds_epoch = np.concatenate(train_preds_epoch)
        train_true_epoch = np.concatenate(train_true_epoch)
        train_rmse = float(np.sqrt(mean_squared_error(train_true_epoch, train_preds_epoch)))

        model.eval()
        val_preds_epoch = []
        val_true_epoch = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb)
                val_preds_epoch.append(preds.cpu().numpy() * RUL_CAP)
                val_true_epoch.append(yb.cpu().numpy() * RUL_CAP)

        val_preds_epoch = np.concatenate(val_preds_epoch)
        val_true_epoch = np.concatenate(val_true_epoch)
        val_rmse = float(np.sqrt(mean_squared_error(val_true_epoch, val_preds_epoch)))

        train_rmse_hist.append(train_rmse)
        val_rmse_hist.append(val_rmse)

        # Save best checkpoint using validation RMSE.
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            no_improve = 0
            torch.save(model.state_dict(), MODEL_DIR / "best_model.pth")
        else:
            no_improve += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:02d} | Train RMSE: {train_rmse:.4f} | Val RMSE: {val_rmse:.4f}")

        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Best validation RMSE: {best_val_rmse:.4f}")

    plt.figure(figsize=(8, 4))
    plt.plot(train_rmse_hist, label="Train RMSE", color="#2563eb")
    plt.plot(val_rmse_hist, label="Val RMSE", color="#f59e0b")
    plt.xlabel("Epoch")
    plt.ylabel("RMSE")
    plt.title("FD004 Training Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "training_loss_curve.png", dpi=150)

    best_model = LSTMRULModel(input_size=input_size, hidden_size=128, num_layers=2, dropout=0.2).to(DEVICE)
    best_model.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location=DEVICE))
    best_model.eval()

    with torch.no_grad():
        val_preds = best_model(torch.tensor(X_val, dtype=torch.float32, device=DEVICE)).cpu().numpy() * RUL_CAP

    val_rmse = float(np.sqrt(mean_squared_error(y_val, val_preds)))
    print(f"Validation RMSE (real scale): {val_rmse:.4f}")
    print(f"Prediction STD: {float(np.std(val_preds)):.4f}")

    plt.figure(figsize=(6, 6))
    plt.scatter(y_val, val_preds, alpha=0.45, s=12, color="#f59e0b")
    lim = max(float(np.max(y_val)), float(np.max(val_preds)))
    plt.plot([0, lim], [0, lim], "r--", linewidth=1.4)
    plt.xlabel("Actual RUL")
    plt.ylabel("Predicted RUL")
    plt.title("Validation: Predicted vs Actual")
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "predicted_vs_actual_rul.png", dpi=150)


if __name__ == "__main__":
    main()

## Evaluation

In [ ]:
import json
import pickle
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path(__file__).resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from model.lstm_model import LSTMRULModel


PROCESSED_DIR = Path("data/processed")
MODEL_PATH = Path("model/best_model.pth")
DEVICE = torch.device("cpu")
RUL_CAP = 125.0


def main() -> None:
    # Load final test sequences and saved trajectories.
    X_test = np.load(PROCESSED_DIR / "X_test_last.npy")
    y_test = np.load(PROCESSED_DIR / "y_test_last.npy")
    test_unit_ids = np.load(PROCESSED_DIR / "test_unit_ids.npy")

    with open(PROCESSED_DIR / "test_engine_trajectories.pkl", "rb") as fp:
        trajectories = pickle.load(fp)

    input_size = X_test.shape[-1]
    model = LSTMRULModel(input_size=input_size, hidden_size=128, num_layers=2, dropout=0.2).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    with torch.no_grad():
        preds = model(torch.tensor(X_test, dtype=torch.float32, device=DEVICE)).cpu().numpy() * RUL_CAP

    # Compute standard regression metrics.
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    mae = float(mean_absolute_error(y_test, preds))
    r2 = float(r2_score(y_test, preds))

    print(f"RMSE: {rmse:.4f}")
    print(f"MAE : {mae:.4f}")
    print(f"R2  : {r2:.4f}")
    print(f"Prediction STD: {float(np.std(preds)):.4f}")

    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, preds, alpha=0.6, s=18, color="#f59e0b")
    lim = max(float(np.max(y_test)), float(np.max(preds)))
    plt.plot([0, lim], [0, lim], "r--", linewidth=1.4)
    plt.xlabel("Actual RUL")
    plt.ylabel("Predicted RUL")
    plt.title("FD004 Test: Predicted vs Actual")
    plt.tight_layout()
    plt.savefig("model/test_pred_vs_actual.png", dpi=150)

    # Pick three engines: high, middle, and low final RUL.
    final_pairs = sorted([(uid, trajectories[int(uid)]["final_rul"]) for uid in test_unit_ids], key=lambda x: x[1])
    selected = [int(final_pairs[-1][0]), int(final_pairs[len(final_pairs) // 2][0]), int(final_pairs[0][0])]

    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)
    for ax, engine_id in zip(axes, selected):
        item = trajectories[engine_id]
        seqs = torch.tensor(item["sequences"], dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            pred_traj = model(seqs).cpu().numpy() * RUL_CAP

        cycles = np.array(item["cycles"])
        actual = np.array(item["actual_rul"])

        ax.plot(cycles, actual, label="Actual", color="#2563eb", linewidth=1.5)
        ax.plot(cycles, pred_traj, label="Predicted", color="#f59e0b", linewidth=1.5)
        ax.set_ylabel("RUL")
        ax.set_title(f"Engine {engine_id} trajectory")
        ax.legend(loc="upper right")

    axes[-1].set_xlabel("Cycle")
    plt.tight_layout()
    plt.savefig("model/test_trajectory_examples.png", dpi=150)

    with open(PROCESSED_DIR / "latest_test_metrics.json", "w", encoding="utf-8") as fp:
        json.dump({"rmse": rmse, "mae": mae, "r2": r2, "prediction_std": float(np.std(preds))}, fp, indent=2)


if __name__ == "__main__":
    main()

## Inference

In [ ]:
import pickle
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path(__file__).resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from model.lstm_model import LSTMRULModel, mc_dropout_predict


PROCESSED_DIR = Path("data/processed")
MODEL_PATH = Path("model/best_model.pth")
DEVICE = torch.device("cpu")
RUL_CAP = 125.0


def main() -> None:
    # Load stored trajectories and choose one engine for demo.
    with open(PROCESSED_DIR / "test_engine_trajectories.pkl", "rb") as fp:
        trajectories = pickle.load(fp)

    engine_ids = sorted(trajectories.keys())
    engine_id = engine_ids[len(engine_ids) // 2]
    payload = trajectories[engine_id]

    input_size = payload["sequences"].shape[-1]
    model = LSTMRULModel(input_size=input_size, hidden_size=128, num_layers=2, dropout=0.2).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))

    sequences = torch.tensor(payload["sequences"], dtype=torch.float32, device=DEVICE)
    cycles = np.array(payload["cycles"])
    actual = np.array(payload["actual_rul"])

    # Monte Carlo dropout gives mean prediction + uncertainty.
    mean, std = mc_dropout_predict(model, sequences, n_passes=50)
    mean = mean.numpy() * RUL_CAP
    std = std.numpy() * RUL_CAP

    lower = mean - 1.96 * std
    upper = mean + 1.96 * std

    plt.figure(figsize=(12, 5))
    plt.plot(cycles, actual, label="Actual RUL", color="#2563eb", linewidth=1.7)
    plt.plot(cycles, mean, label="Predicted Mean", color="#f59e0b", linewidth=1.7)
    plt.fill_between(cycles, lower, upper, color="#f59e0b", alpha=0.25, label="95% CI")
    plt.xlabel("Cycle")
    plt.ylabel("RUL")
    plt.title(f"FD004 Engine {engine_id}: RUL Inference with Uncertainty")
    plt.legend()
    plt.tight_layout()
    plt.savefig("model/inference_uncertainty_curve.png", dpi=150)

    final_mean = float(mean[-1])
    final_std = float(std[-1])
    print(f"Engine ID: {engine_id}")
    print(f"Final predicted RUL: {final_mean:.2f}")
    print(f"Final uncertainty (std): {final_std:.2f}")
    print(f"Final 95% CI: [{(final_mean - 1.96 * final_std):.2f}, {(final_mean + 1.96 * final_std):.2f}]")


if __name__ == "__main__":
    main()

## API

In [ ]:
import csv
import io
import json
from pathlib import Path
from typing import List

import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, field_validator

from model.lstm_model import LSTMRULModel, mc_dropout_predict


PROJECT_ROOT = Path(__file__).resolve().parent.parent
FEATURE_COLUMNS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_columns.json"
RUL_CAP = 125.0

FEATURE_COUNT = 15
if FEATURE_COLUMNS_PATH.exists():
    with open(FEATURE_COLUMNS_PATH, "r", encoding="utf-8") as fp:
        FEATURE_COUNT = len(json.load(fp))


class PredictRequest(BaseModel):
    # Client can send either parsed sequence or raw CSV text.
    engine_id: int = Field(0, ge=0)
    sequence: List[List[float]] | None = None
    csv_data: str | None = None

    @field_validator("sequence", mode="before")
    @classmethod
    def validate_sequence_shape(cls, value):
        if value is None:
            return value
        if len(value) != 30:
            raise ValueError("sequence must have exactly 30 timesteps")
        for row in value:
            if len(row) != FEATURE_COUNT:
                raise ValueError(f"each timestep must contain exactly {FEATURE_COUNT} features")
        return value


class PredictResponse(BaseModel):
    rul: float
    uncertainty: float
    confidence_interval_95: List[float]


app = FastAPI(title="FD004 RUL Inference API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5500", "http://127.0.0.1:5500"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

MODEL_PATH = PROJECT_ROOT / "model" / "best_model.pth"
MODEL = None
DEVICE = torch.device("cpu")


def parse_csv_sequence(csv_data: str) -> List[List[float]]:
    # Parse CSV text into numeric 30 x FEATURE_COUNT matrix.
    rows: List[List[float]] = []
    reader = csv.reader(io.StringIO(csv_data.strip()))
    for line_no, row in enumerate(reader, start=1):
        cleaned = [col.strip() for col in row if col.strip()]
        if not cleaned:
            continue

        try:
            parsed = [float(v) for v in cleaned]
        except ValueError as exc:
            if not rows:
                continue
            raise ValueError(
                f"csv_data row {line_no} contains non-numeric values. "
                "Use only numeric values after the optional header row."
            ) from exc

        rows.append(parsed)

    if len(rows) != 30:
        raise ValueError("csv_data must contain exactly 30 rows")
    for i, row in enumerate(rows, start=1):
        if len(row) != FEATURE_COUNT:
            raise ValueError(
                f"each csv_data row must contain exactly {FEATURE_COUNT} values; "
                f"row {i} has {len(row)}"
            )
    return rows


@app.on_event("startup")
def load_model_once() -> None:
    # Load model once when API starts.
    global MODEL
    model = LSTMRULModel(input_size=FEATURE_COUNT, hidden_size=128, num_layers=2, dropout=0.2)
    if not MODEL_PATH.exists():
        MODEL = None
        return

    state = torch.load(MODEL_PATH, map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    MODEL = model


@app.get("/health")
def health() -> dict:
    return {"status": "ok", "dataset": "FD004", "feature_count": FEATURE_COUNT}


@app.post("/predict_rul", response_model=PredictResponse)
def predict_rul(payload: PredictRequest) -> PredictResponse:
    if MODEL is None:
        raise HTTPException(status_code=503, detail="Model checkpoint missing. Run training first.")

    if payload.sequence is None and payload.csv_data is None:
        raise HTTPException(status_code=422, detail="Provide either 'sequence' or 'csv_data'.")

    try:
        sequence = payload.sequence if payload.sequence is not None else parse_csv_sequence(payload.csv_data or "")
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc

    x = torch.tensor([sequence], dtype=torch.float32, device=DEVICE)
    mean, std = mc_dropout_predict(MODEL, x, n_passes=50)

    rul = float(mean.item()) * RUL_CAP
    unc = float(std.item()) * RUL_CAP
    ci_low = rul - 1.96 * unc
    ci_high = rul + 1.96 * unc

    return PredictResponse(
        rul=round(rul, 1),
        uncertainty=round(unc, 2),
        confidence_interval_95=[round(ci_low, 1), round(ci_high, 1)],
    )